# What
分类任务，支持两种模式
1. Folder模式，需要输入`train`, `valid`两个测试集对应的目录。`labels.txt`，需要训练的label，里面每个类别一行。
2. List模式，需要输入`train`, `valid`两个测试集对应的训练文件，每行一个样本。`labels.txt`是可选参数，里面每个类别一行。`data_pattern`一个通用的目录，与train、val中的第一列进行拼接。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=512, nhead=8, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        x = x + self.dropout1(attn_out)
        x = self.norm1(x)

        ff_out = self.linear2(self.dropout2(self.activation(self.linear1(x))))
        x = x + ff_out
        x = self.norm2(x)
        return x


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim=512, num_classes=2,
                 d_model=512, nhead=8, num_layers=3,
                 dim_feedforward=2048, dropout=0.1):
        super().__init__()
        if input_dim != d_model:
            self.proj = nn.Linear(input_dim, d_model)
        else:
            self.proj = nn.Identity()

        self.encoder = nn.Sequential(*[
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)

        x = self.proj(x)
        x = self.encoder(x)
        x = x[:, 0, :]
        out = self.fc(x)
        return out


def train_model(model, train_loader, val_loader, device,
                epochs=100, lr=0.001, momentum=0.9, weight_decay=1e-4,
                patience=15, save_path='best_model.pth'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss /= total
        train_acc = 100. * correct / total

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss /= total
        val_acc = 100. * correct / total

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_wts = model.state_dict().copy()
            torch.save(best_model_wts, save_path)
            print(f"  -> Best model saved (val_loss={val_loss:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break

    model.load_state_dict(best_model_wts)
    return model


def main():
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    batch_size = 128
    epochs = 100
    lr = 0.001
    momentum = 0.9
    num_classes = 2
    input_dim = 512
    patience = 15

    train_feat_path = 'train_features.npy'
    train_label_path = 'train_labels.npy'
    val_feat_path = 'val_features.npy'
    val_label_path = 'val_labels.npy'
    test_feat_paths = ['test1_features.npy', 'test2_features.npy']
    test_label_paths = ['test1_labels.npy', 'test2_labels.npy']

    try:
        train_feat = np.load(train_feat_path).astype(np.float32)
        train_label = np.load(train_label_path).astype(np.int64)
        val_feat = np.load(val_feat_path).astype(np.float32)
        val_label = np.load(val_label_path).astype(np.int64)
    except FileNotFoundError as e:
        print(f"文件未找到，请检查路径。错误: {e}")
        print("使用随机数据演示。")
        num_train, num_val = 1000, 200
        train_feat = np.random.randn(num_train, input_dim).astype(np.float32)
        train_label = np.random.randint(0, 2, size=(num_train,))
        val_feat = np.random.randn(num_val, input_dim).astype(np.float32)
        val_label = np.random.randint(0, 2, size=(num_val,))

    train_dataset = TensorDataset(torch.from_numpy(train_feat), torch.from_numpy(train_label))
    val_dataset = TensorDataset(torch.from_numpy(val_feat), torch.from_numpy(val_label))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    model = TransformerClassifier(input_dim=input_dim, num_classes=num_classes).to(device)

    print("开始训练...")
    model = train_model(model, train_loader, val_loader, device,
                        epochs=epochs, lr=lr, momentum=momentum,
                        patience=patience, save_path='best_transformer.pth')

    print("\n测试结果:")
    model.eval()
    for i, (test_feat_path, test_label_path) in enumerate(zip(test_feat_paths, test_label_paths)):
        try:
            test_feat = np.load(test_feat_path).astype(np.float32)
            test_label = np.load(test_label_path).astype(np.int64)
        except FileNotFoundError:
            print(f"测试文件 {test_feat_path} 未找到，跳过。")
            continue

        test_dataset = TensorDataset(torch.from_numpy(test_feat), torch.from_numpy(test_label))
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f"Test set {i+1}: Accuracy = {acc:.2f}% ({correct}/{total})")


if __name__ == '__main__':
    main()